# 05 — Cost-sensitive decision (German Credit cost matrix)

**Objective.** Replace the naive `defaults_avoided` metric with the official cost matrix of the
German Credit dataset (UCI / Statlog) and derive the optimal decision threshold from it.

Cost matrix:

|                     | Accept | Reject |
|---------------------|:------:|:------:|
| **Good customer**   | 0      | 1      |
| **Bad customer**    | 5      | 0      |

Accepting a bad customer costs 5x more than rejecting a good one.

In [ ]:
import sys, json, os
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cost_matrix import (
    COST_ACCEPT_BAD, COST_REJECT_GOOD, expected_cost, optimal_pd_threshold,
    cost_by_threshold, stress_cost, total_portfolio_cost,
)
from decision import score_to_pd, pd_to_score, optimal_score_threshold

params = json.load(open("../models/scaling_params.json"))
OFFSET, FACTOR, INTERCEPT = params["Offset"], params["Factor"], params["intercept"]

print(f"Costs -> reject good: {COST_REJECT_GOOD}, accept bad: {COST_ACCEPT_BAD}")
print(f"Factor={FACTOR:.4f}, Offset={OFFSET:.4f}, intercept={INTERCEPT:.4f}")

## 1. The cost matrix

Visualising the asymmetric cost structure. The worst outcome is accepting a bad customer (5 units).

In [ ]:
matrix = np.array([[0, COST_REJECT_GOOD], [COST_ACCEPT_BAD, 0]])

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(matrix, cmap="RdYlGn_r")
ax.set_xticks([0, 1], ["Accept", "Reject"])
ax.set_yticks([0, 1], ["Good", "Bad"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(matrix[i, j]), ha="center", va="center", fontsize=16)
plt.colorbar(im, ax=ax, label="Cost (units)")
plt.title("German Credit - official cost matrix")
plt.tight_layout()
os.makedirs("../models", exist_ok=True)
plt.savefig("../models/cost_matrix.png", dpi=120)
plt.show()

## 2. Deriving the break-even PD

Decision rule:

```
cost(accept) = 5 * PD
cost(reject) = 1 * (1 - PD)

accept if 5*PD < 1-PD  <=>  6*PD < 1  <=>  PD < 1/6 ~ 16.67%
```

This replaces arbitrary cutoffs with a mathematically derived threshold.

In [ ]:
t = optimal_pd_threshold()
print(f"Optimal PD threshold : {t:.4f} ({t:.2%})")
print(f"cost(accept) at t    : {expected_cost(t, True):.4f}")
print(f"cost(reject) at t    : {expected_cost(t, False):.4f}")
print(f"-> break-even verified: {np.isclose(expected_cost(t, True), expected_cost(t, False))}")

score_cutoff = optimal_score_threshold(OFFSET, FACTOR, INTERCEPT)
print(f"\nOptimal score cutoff: {score_cutoff:.2f}")
print(f"-> accept when score > {score_cutoff:.2f}")

strategy = json.load(open("../models/decision_strategy.json"))
print("\nBusiness baseline cutoffs:", strategy["cutoffs"])

## 3. Portfolio cost by threshold

We convert every client's score into PD, then sweep several PD cutoffs and compute the **total cost**
of the resulting decisions. This is the metric that replaces `defaults_avoided`.

In [ ]:
test_df = pd.read_parquet("../data/processed/test_scores_with_target.parquet").copy()
test_df["pd"] = score_to_pd(test_df["score"], OFFSET, FACTOR, INTERCEPT)

print(test_df[["default", "score", "pd"]].head().to_string())
print(f"\nMean PD: {test_df['pd'].mean():.4f} | observed default rate: {test_df['default'].mean():.4f}")

In [ ]:
thresholds = np.round(np.arange(0.02, 0.51, 0.02), 3)
cost_df = cost_by_threshold(test_df["default"], test_df["pd"], thresholds)

best = cost_df.loc[cost_df["total_cost"].idxmin()]
print(cost_df.head(10).to_string(index=False))
print(f"\nMinimum cost at PD < {best['threshold_pd']:.3f} -> total cost = {best['total_cost']:.0f} units")
print(f"Official optimum PD < {t:.3f} -> total cost = "
      f"{total_portfolio_cost(test_df['default'], test_df['pd'], test_df['pd'] < t):.0f} units")

## 4. Cost curve

The minimum of this curve is the economically optimal cutoff. The gap between the empirical minimum
and the theoretical optimum (1/6) is a validation check of the scorecard calibration.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(cost_df["threshold_pd"], cost_df["total_cost"], marker="o", label="Portfolio cost")
ax.axvline(t, color="red", linestyle="--", label=f"Theoretical optimum (PD={t:.3f})")
ax.axvline(best["threshold_pd"], color="green", linestyle="--", label=f"Empirical min (PD={best['threshold_pd']:.3f})")
ax.set_xlabel("PD cutoff")
ax.set_ylabel("Total cost (units)")
ax.set_title("Cost curve - optimal decision threshold")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../models/cost_curve.png", dpi=120)
plt.show()

## 5. Stress test — downturn scenario

A downturn is simulated by turning 20% of the **accepted good clients** into defaults. The decision policy stays fixed; we measure the additional cost.


In [ ]:
approved = test_df["pd"] < t
stress = stress_cost(test_df["default"], test_df["pd"], approved, shock=0.20)

for k, v in stress.items():
    print(f"{k:28s}: {v:.4f}" if isinstance(v, float) else f"{k:28s}: {v}")
print(f"\nCost increase under downturn: "
      f"{(stress['stressed_cost'] / stress['baseline_cost'] - 1):.1%}")

## 6. Save artifacts

The derived threshold and the cost matrix are persisted for the decision engine and the report.

In [ ]:
with open("../models/cost_matrix.json", "w") as f:
    json.dump({
        "cost_reject_good": COST_REJECT_GOOD,
        "cost_accept_bad": COST_ACCEPT_BAD,
        "source": "UCI / Statlog German Credit dataset",
    }, f, indent=2)

with open("../models/optimal_threshold.json", "w") as f:
    json.dump({
        "optimal_pd": round(t, 4),
        "optimal_score": round(float(score_cutoff), 2),
        "rule": "accept if score > optimal_score",
        "derivation": "5*PD = 1*(1-PD) -> PD = 1/6",
    }, f, indent=2)

print("Saved models/cost_matrix.json and models/optimal_threshold.json")